In [ ]:
import numpy as np
from numpy import dot, exp
import pandas as pd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.metrics import log_loss
import copy

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer

from sklearn.model_selection import KFold , GridSearchCV,StratifiedKFold
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')

In [ ]:
"""
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
!pip install tabpfn --no-index --find-links=file:///kaggle/input/tabpfn
!mkdir -p /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff
!cp /kaggle/input/tabpfn/prior_diff_real_checkpoint_n_0_epoch_100.cpkt /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff/
from tabpfn import TabPFNClassifier
"""

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
int_denominators = {
 'AB': 0.004273,
 'AF': 0.00242,
 'AH': 0.008709,
 'AM': 0.003097,
 'AR': 0.005244,
 'AX': 0.008859,
 'AY': 0.000609,
 'AZ': 0.006302,
 'BC': 0.007028,
 'BD ': 0.00799,
 'BN': 0.3531,
 'BP': 0.004239,
 'BQ': 0.002605,
 'BR': 0.006049,
 'BZ': 0.004267,
 'CB': 0.009191,
 'CC': 6.12e-06,
 'CD ': 0.007928,
 'CF': 0.003041,
 'CH': 0.000398,
 'CL': 0.006365,
 'CR': 7.5e-05,
 'CS': 0.003487,
 'CU': 0.005517,
 'CW ': 9.2e-05,
 'DA': 0.00388,
 'DE': 0.004435,
 'DF': 0.000351,
 'DH': 0.002733,
 'DI': 0.003765,
 'DL': 0.00212,
 'DN': 0.003412,
 'DU': 0.0013794,
 'DV': 0.00259,
 'DY': 0.004492,
 'EB': 0.007068,
 'EE': 0.004031,
 'EG': 0.006025,
 'EH': 0.006084,
 'EL': 0.000429,
 'EP': 0.009269,
 'EU': 0.005064,
 'FC': 0.005712,
 'FD ': 0.005937,
 'FE': 0.007486,
 'FI': 0.005513,
 'FR': 0.00058,
 'FS': 0.006773,
 'GB': 0.009302,
 'GE': 0.004417,
 'GF': 0.004374,
 'GH': 0.003721,
 'GI': 0.002572
}
for k, v in int_denominators.items():
    train[k] = np.round(train[k]/v,1)
    test[k] = np.round(test[k]/v,1)

In [ ]:
BN_mapper = train.groupby('BN').Class.mean().to_dict()

In [ ]:
train['BN_TE'] = train['BN'].map(BN_mapper)
test['BN_TE'] = test['BN'].map(BN_mapper)

In [ ]:
pd.to_datetime(greeks['Epsilon'].apply(lambda x : np.nan if x=='Unknown' else x)).max()

In [ ]:
((pd.to_datetime(greeks['Epsilon'].apply(lambda x : np.nan if x=='Unknown' else x))>'2020-03-01') & (pd.to_datetime(greeks['Epsilon'].apply(lambda x : np.nan if x=='Unknown' else x))<'2020-08-01') )*1

In [ ]:
train['magic'] = ((pd.to_datetime(greeks['Epsilon'].apply(lambda x : np.nan if x=='Unknown' else x))>'2020-03-01')*1)#.sum()
test['magic'] = 1

In [ ]:
train['year'] = greeks.apply(lambda x: -1 if x['Epsilon'] == 'Unknown' else int(x['Epsilon'].split('/')[-1]), axis=1)

In [ ]:
test['year'] = 2021

In [ ]:
corr_matrix = train.corr()

# Set the correlation threshold
threshold = 0.8

# Find highly correlated features
highly_correlated = (corr_matrix.abs() > threshold)

# Get the column names of highly correlated features
correlated_features = set()

for i in range(len(highly_correlated.columns)):
    for j in range(i):
        if highly_correlated.iloc[i, j]:
            colname = highly_correlated.columns[i]
            correlated_features.add(colname)

print("Highly correlated features:")
for feature in correlated_features:
    print(feature)

In [ ]:
corr_matrix = train.corr()

correlated_pairs = []

for i in range(len(highly_correlated.columns)):
    for j in range(i):
        if highly_correlated.iloc[i, j]:
            pair = (highly_correlated.columns[i], highly_correlated.columns[j])
            correlation = corr_matrix.iloc[i, j]
            correlated_pairs.append((pair, correlation))

print("Pairs of highly correlated features and their correlation values:")
for pair, correlation in correlated_pairs:
    print(pair, correlation)

In [ ]:
train['EJ'] = train.EJ.map({'A':0,'B':1})
test['EJ'] = test.EJ.map({'A':0,'B':1})

In [ ]:
FEATURES = train.columns.difference(['Class','Id','EH','DV','GL','BZ']).tolist()

In [ ]:
train.columns

In [ ]:
train['BQ_NONE'] = train['BQ'].isna().astype(int)
test['BQ_NONE'] = test['BQ'].isna().astype(int)

FEATURES = ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 'BD ', 'BN',
            'BP', 'BQ', 'BR', 'BZ', 'CB', 'CC', 'CD ', 'CF', 'CH', 'CL', 'CR', 'CS',
            'CU', 'CW ', 'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
            'EB', 'EE', 'EG', 'EH', 'EJ', 'EL', 'EP', 'EU', 'FC', 'FD ', 'FE', 'FI',
            'FL', 'FR', 'FS', 'GB', 'GE', 'GF', 'GH', 'GI', 'GL', 'BQ_NONE', 'BN_TE', 'magic']

In [ ]:
def balance_logloss(y_true, y_pred):

    y_pred = np.stack([1-y_pred,y_pred]).T
    y_pred = np.clip(y_pred, 1e-15, 1-1e-15)
    y_pred / np.sum(y_pred, axis=1)[:, None]
    nc = np.bincount(y_true)

    logloss = (-1/nc[0]*(np.sum(np.where(y_true==0,1,0) * np.log(y_pred[:,0]))) - 1/nc[1]*(np.sum(np.where(y_true!=0,1,0) * np.log(y_pred[:,1])))) / 2

    return logloss

In [ ]:
LGB_Features = ['AB','AF','AH','AM','AR','AX','AY','AZ','BC','BD ','BN','BN_TE','BP','BQ','BR','CB','CC',
                'CD ','CF','CH','CL','CR','CS','CU','CW ','DA','DE','DF','DH','DI','DL','DN','DU','DY',
                'EB','EE','EG','EJ','EL','EP','EU','FC','FD ','FE','FI','FL','FR','FS','GB','GE','GF','GH',
                'GI','BQ_NONE','magic']

XGB_Features = ['AB','AF','AH','AM','AR','AX','AY','AZ','BC','BD ','BN','BN_TE','BP','BQ','BR','CB','CC',
                'CD ','CF','CH','CL','CR','CS','CU','CW ','DA','DE','DF','DH','DI','DL','DN','DU','DY',
                'EB','EE','EG','EJ','EL','EP','EU','FC','FD ','FE','FI','FL','FR','FS','GB','GE','GF','GH',
                'GI','BQ_NONE','magic']

SVC_Features = ['AB', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 'BD ', 'BQ', 'BR', 'CB', 'CC',
                'CD ', 'CF', 'CH', 'CL', 'CR', 'CS', 'CU', 'CW ', 'DF', 'DH', 'DI', 'DL', 
                'DN', 'DU', 'DV', 'DY', 'EB', 'EE', 'EG', 'EH', 'EJ', 'EL', 'FC', 'FD ', 
                'FE', 'FI', 'FL', 'FR', 'FS', 'GF', 'GH', 'BQ_NONE', 'BN_TE', 'magic']

In [ ]:
BAGS = 20
FOLDS = 20
oof = np.zeros(len(train))
models = {}
oof2 = np.zeros(len(train))
models2 = {}
#oof3 = np.zeros(len(train))
#models3 = {}
oof5 = np.zeros(len(train))
models5 = {}

feature_importance_LGB = {}
feature_importance_XGB = {}
feature_importance_SVC = {}

for bag in range(BAGS):
    print('#'*25)
    print('### Bag',bag+1)
    print('#'*25)
    models[bag] = []
    models2[bag] = []
    #models3[bag] = []
    models5[bag] = []
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=bag)
    for fold,(train_idx, valid_idx) in enumerate(skf.split(X=train[FEATURES], y=greeks['Alpha'])):

        print('=> Fold',fold+1,', ',end='')

        # DOWNSAMPLE NEGATIVE CLASS TO BALANCE CLASSES
        y_train = train.loc[train_idx,'Class']
        RMV = y_train.loc[y_train==0].sample(
            frac=0.7, random_state=bag*BAGS+fold).index.values
        train_idx = np.setdiff1d(train_idx,RMV)

        # TRAIN DATA
        X_train = train.loc[train_idx,FEATURES]
        y_train = train.loc[train_idx,'Class']

        # VALID DATA
        X_valid = train.loc[valid_idx,FEATURES]
        y_valid = train.loc[valid_idx,'Class']

        # TRAIN MODEL
        clf = LGBMClassifier(n_estimators=800,
                             max_depth=9,
                             learning_rate=0.175, 
                             metric='cross_entropy_lambda',
                             objective=None,
                             boosting_type='gbdt',
                             num_leaves=31,
                             reg_alpha=0.0,
                             reg_lambda=0.0000091,
                             early_stopping_round=50,
                             feature_fraction=0.2,
                             class_weight='balanced')
        clf.fit(X_train[LGB_Features], y_train,eval_set=[(X_train[LGB_Features],y_train),(X_valid[LGB_Features], y_valid)], verbose=100)
        importance_LGB = clf.feature_importances_
        feature_importance_LGB[bag] = importance_LGB
        
        positive_class_ratio = np.sum(y_train == 0) / np.sum(y_train == 1)
        clf2 = XGBClassifier(
            n_estimators=800,
            learning_rate=0.176,
            objective='binary:logistic',
            booster='gbtree',
            max_depth=9,
            subsample=0.8,
            colsample_bytree=0.3,
            reg_alpha=0.0,
            reg_lambda=0.0000088,
            scale_pos_weight=positive_class_ratio,
            early_stopping_rounds=50)
        clf2.fit(X_train[XGB_Features], y_train,eval_set=[(X_train[XGB_Features], y_train),(X_valid[XGB_Features], y_valid)], verbose=100)
        importance_XGB = clf2.feature_importances_
        feature_importance_XGB[bag] = importance_XGB
        
        #clf3 = TabPFNClassifier(device='cpu',N_ensemble_configurations=4)
        #clf3.fit(X_train, y_train)
        
        imputer = SimpleImputer(strategy='median')
        X_train_imputed_SVC = imputer.fit_transform(X_train[SVC_Features])
        X_valid_imputed_SVC = imputer.transform(X_valid[SVC_Features])
        scaler = StandardScaler()
        X_train_scaled_SVC = scaler.fit_transform(X_train_imputed_SVC)
        X_valid_scaled_SVC = scaler.fit_transform(X_valid_imputed_SVC)
        
        #feature_fraction = 0.9
        #num_features = int(feature_fraction * X_train_scaled_SVC.shape[1])
        #selected_features_indices = np.random.choice(X_train_scaled_SVC.shape[1], num_features, replace=False)
        #X_train_feature_subsampled = X_train_scaled_SVC[:, selected_features_indices]
        #X_valid_feature_subsampled = X_valid_scaled_SVC[:, selected_features_indices]
        clf5 = SVC(
            C=5.0,
            kernel='rbf',
            class_weight='balanced',
            probability=True,
            random_state=bag*BAGS+i)
        clf5.fit(X_train_scaled_SVC, y_train)
        result_SVC = permutation_importance(clf5, X_valid_scaled_SVC, y_valid, n_repeats=10, random_state=bag*BAGS+fold)
        feature_importance_SVC[bag] = result_SVC.importances_mean

        oof[valid_idx] += clf.predict_proba(X_valid[LGB_Features])[:,1] / BAGS
        models[bag].append(clf)
        
        oof2[valid_idx] += clf2.predict_proba(X_valid[XGB_Features])[:,1] / BAGS
        models2[bag].append(clf2)
        
        #oof3[valid_idx] += clf3.predict_proba(X_valid)[:,1] / BAGS
        #models3[bag].append(clf3)
        
        oof5[valid_idx] += clf5.predict_proba(X_valid_scaled_SVC)[:,1] / BAGS
        models5[bag].append(clf5)
        
    print()

In [ ]:
test1 = test.copy()
train1 = train.copy()

In [ ]:
train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')

In [ ]:
greeks['k'] = greeks['Alpha']+greeks['Beta']+greeks['Gamma']+greeks['Delta']
train = pd.merge( greeks[['k','Id']],train,on='Id')

names = ['AB', 'AF', 'AH', 'AM', 'AR', 'AX', 'AY', 'AZ', 'BC', 'BD ', 'BN',
       'BP', 'BQ', 'BR', 'BZ', 'CB', 'CC', 'CD ', 'CF', 'CH', 'CL', 'CR', 'CS',
       'CU', 'CW ', 'DA', 'DE', 'DF', 'DH', 'DI', 'DL', 'DN', 'DU', 'DV', 'DY',
       'EB', 'EE', 'EG', 'EH', 'EJ', 'EL', 'EP', 'EU', 'FC', 'FD ', 'FE', 'FI',
       'FL', 'FR', 'FS', 'GB', 'GE', 'GF', 'GH', 'GI', 'GL']
target_name = 'Class'

In [ ]:
train['EJ'] = pd.Series(np.where(train.EJ.values == 'A', 1, 0),
          train.index)
test['EJ'] = pd.Series(np.where(test.EJ.values == 'A', 1, 0),
          test.index)

# fill nan data with mean values 
train[names] = train[names].fillna(train[names].mean())
test[names] = test[names].fillna(train[names].mean())
# clip values to avoid different values in the test set from train
test = test[names].clip(train[names].min(axis=0).values,train[names].max(axis=0).values, axis=1)

# data scaled to allow the features interaction (by multiplication)
scaler = StandardScaler()

train2 = copy.copy(train)
teste2 = copy.copy(test)

vals = scaler.fit_transform(train[names])
vals_test = scaler.transform(test[names])

train2[names] = vals
teste2[names] = vals_test

In [ ]:
def mab(df,nome1,nome2):
    a  = df[nome1]*df[nome2]
    return(a/max(a))

h = []
ht = []

n = 1
for n1 in names:
    for n2 in names[n:]:
        h.append(mab(train2,n1,n2).rename(n1+'_mul_'+n2))
        ht.append(mab(teste2,n1,n2).rename(n1+'_mul_'+n2))
        
    n+=1
    
newF = pd.DataFrame(h)
newF_test = pd.DataFrame(ht)

In [ ]:
def iv_woe(data, target, bins=10, show_woe=False):
    
    #Empty Dataframe
    newDF,woeDF = pd.DataFrame(), pd.DataFrame()
    
    #Extract Column Names
    cols = data.columns
    
    #Run WOE and IV on all the independent variables
    for ivars in cols[~cols.isin([target])]:
        if (data[ivars].dtype.kind in 'bifc') and (len(np.unique(data[ivars]))>10):
            binned_x = pd.qcut(data[ivars], bins,  duplicates='drop')
            d0 = pd.DataFrame({'x': binned_x, 'y': data[target]})
        else:
            d0 = pd.DataFrame({'x': data[ivars], 'y': data[target]})

        
        # Calculate the number of events in each group (bin)
        d = d0.groupby("x", as_index=False).agg({"y": ["count", "sum"]})
        d.columns = ['Cutoff', 'N', 'Events']
        
        # Calculate % of events in each group.
        d['% of Events'] = np.maximum(d['Events'], 0.5) / d['Events'].sum()

        # Calculate the non events in each group.
        d['Non-Events'] = d['N'] - d['Events']
        # Calculate % of non events in each group.
        d['% of Non-Events'] = np.maximum(d['Non-Events'], 0.5) / d['Non-Events'].sum()

        # Calculate WOE by taking natural log of division of % of non-events and % of events
        d['WoE'] = np.log(d['% of Events']/d['% of Non-Events'])
        d['IV'] = d['WoE'] * (d['% of Events'] - d['% of Non-Events'])
        d.insert(loc=0, column='Variable', value=ivars)
        #print("Information value of " + ivars + " is " + str(round(d['IV'].sum(),6)))
        temp =pd.DataFrame({"Variable" : [ivars], "IV" : [d['IV'].sum()]}, columns = ["Variable", "IV"])
        newDF=pd.concat([newDF,temp], axis=0)
        woeDF=pd.concat([woeDF,d], axis=0)

        #Show WOE Table
        if show_woe == True:
            print(d)
    return newDF, woeDF

In [ ]:
a,b = iv_woe(train2, target_name, bins=10, show_woe=False)

In [ ]:
# most important features based on IV
a.sort_values(by='IV',ascending=False).Variable.values 

In [ ]:
# Reordering the dataframe to keep IV with higger values in front
trainE = train[a.sort_values(by='IV',ascending=False).Variable.values]
trainE[target_name] = train[target_name]
testeE = test[a.sort_values(by='IV',ascending=False).Variable.values[2:]]

# join the original vars and the interactions between them
ff = pd.concat([trainE,newF.T],axis=1)
ff_teste = pd.concat([testeE,newF_test.T],axis=1)

a,b = iv_woe(ff, target_name, bins=10, show_woe=False)

# deleting all IVs below 0.05
a = a.loc[a['IV']> 0.05]

allNames = a.sort_values(by='IV',ascending=False).Variable.values
crossNames = [x for x in allNames if '_mul_' in x]

nomes2 = list(trainE)+crossNames
nomes2.remove('Class')

In [ ]:
# threshold to correlation features
threshold = 0.3

cc = ff[nomes2[2:]].corr()

mat_x = abs(cc)>threshold
mat_x = mat_x.to_numpy()

In [ ]:
# selection tof the variables with low correlation, there are +- 70 features with low correlation
var1 = []
nomes = list(cc)
var1.append(nomes[0])
max_vars = 100

count = 1
for n in range(1,len(nomes)):
    
    if (mat_x[n,:n+1].sum() ) == 1:
        
        var1.append(nomes[n])        
        count+=1
        
        if(count == max_vars):
            break

In [ ]:
# create dic with WoE transformation
list_dics = []

for var in var1:
    df_temp = b.loc[b['Variable']==var].reset_index()
    # criando dicionario
    dict_var = {}
    for x in range(len(df_temp)):
        line = df_temp.iloc[x]
        dict_var[ line['Cutoff'] ] = line['WoE']
    list_dics.append(dict_var)

In [ ]:
# train and test data
df_original = ff[var1+[target_name]+ ['k']]
df_teste2 = ff_teste[var1]
names = var1

In [ ]:
df_original["BQ_None"] = (train.BQ.isna()*1)
df_teste2["BQ_None"] = (test.BQ.isna()*1)

In [ ]:
df_original['magic'] = ((pd.to_datetime(greeks['Epsilon'].apply(lambda x : np.nan if x=='Unknown' else x))>'2020-03-01')*1)#.sum()
df_teste2['magic'] = 1
BN_mapper = train.groupby('BN').Class.mean().to_dict()
df_original['BN_TE'] = df_original['BN'].map(BN_mapper)
df_teste2['BN_TE'] = df_teste2['BN'].map(BN_mapper)

In [ ]:
# In this part there is some data leakage as the map is using the full dataset
n= 0

for var in var1:
    df_original.loc[:,var] = df_original[var].map(list_dics[n])
    df_teste2.loc[:,var] = df_teste2[var].map(list_dics[n])
    n=n+1

In [ ]:
names.extend(["BN_TE", 'magic', "BQ_None"])

In [ ]:
df_original.loc[:,names] = df_original[names].fillna(df_original[names].mean())
df_teste2.loc[:,names] = df_teste2[names].fillna(df_original[names].mean())

In [ ]:
BAGS = 20
FOLDS = 20
oof4 = np.zeros(len(train))

predictions_LR = 0
cv_score_LR = 0
train_score_LR = 0

feature_importance_LR = {}
models4 = {}

for bag in range(BAGS):
    print('#'*25)
    print('### Bag',bag+1)
    print('#'*25)
    models4[bag] = []
    skf = StratifiedKFold(n_splits=FOLDS,shuffle=True, random_state=bag)
    
    for i, (train_idx, valid_idx) in enumerate(skf.split(df_original[target_name], df_original['k'])):
            
            print('=> Fold',i+1,', ',end='')

            # DOWNSAMPLE NEGATIVE CLASS TO BALANCE CLASSES
            y_train = df_original.loc[train_idx,'Class']
            RMV = y_train.loc[y_train==0].sample(
            frac=0.7, random_state=bag*BAGS+i).index.values
            train_idx = np.setdiff1d(train_idx,RMV)

            # TRAIN DATA
            X_train = df_original.loc[train_idx,names]
            y_train = df_original.loc[train_idx,'Class']

            # VALID DATA
            X_valid = df_original.loc[valid_idx,names]
            y_valid = df_original.loc[valid_idx,'Class']        
            
            clf4 = LogisticRegression(
                random_state=bag*BAGS+i,
                C=0.31,
                n_jobs=-1,
                max_iter=2000,
                class_weight='balanced')
            clf4.fit( X_train, y_train) 
            coefficients_LR = clf4.coef_[0]
            feature_importance_LR[bag] = np.abs(coefficients_LR)

            oof4[valid_idx] += clf4.predict_proba(X_valid)[:,1] / BAGS
            models4[bag].append(clf4)
    print()

## stacking

In [ ]:
#df_original['lgb'] = oof
#df_original['xgb'] = oof2
#df_original['svc'] = oof5
#df_original['lr'] = oof4

train1['lgb'] = oof
train1['xgb'] = oof2
train1['svc'] = oof5
train1['lr'] = oof4

In [ ]:
# FE
#df_original['oofs_std'] = df_original[['lgb','xgb', 'svc', 'lr']].apply(lambda x : np.std(x),axis=1)
#df_original['oofs_mean'] = df_original[['lgb','xgb', 'svc', 'lr']].apply(lambda x : np.mean(x),axis=1)
#df_original['oofs_median'] = df_original[['lgb','xgb', 'svc', 'lr']].apply(lambda x : np.median(x),axis=1)
#df_original['oofs_min'] = df_original[['lgb','xgb', 'svc', 'lr']].apply(lambda x : np.min(x),axis=1)
#df_original['oofs_max'] = df_original[['lgb','xgb', 'svc', 'lr']].apply(lambda x : np.max(x),axis=1)

In [ ]:
#names.extend(['lgb','xgb','svc','lr'])

In [ ]:
#names2 = names + ['lgb','xgb','lr', 'svc', 'oofs_std',]

In [ ]:
LGB_Features2 = LGB_Features + ['lgb','xgb','lr', 'svc',]

In [ ]:
BAGS = 20
FOLDS = 20
ensemble = np.zeros(len(train))

predictions_LR = 0
cv_score_LR = 0
train_score_LR = 0

feature_importance_Ensemble = {}
models_final = {}
for bag in range(BAGS):
    print('#'*25)
    print('### Bag',bag+1)
    print('#'*25)
    models_final[bag] = []
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=bag)
    for fold,(train_idx, valid_idx) in enumerate(skf.split(X=train1[LGB_Features2], y=greeks['Alpha'])):

        print('=> Fold',fold+1,', ',end='')

        # DOWNSAMPLE NEGATIVE CLASS TO BALANCE CLASSES
        y_train = train1.loc[train_idx,'Class']
        RMV = y_train.loc[y_train==0].sample(
            frac=0.7, random_state=bag*BAGS+fold).index.values
        train_idx = np.setdiff1d(train_idx,RMV)

        # TRAIN DATA
        X_train = train1.loc[train_idx,LGB_Features2]
        y_train = train1.loc[train_idx,'Class']

        # VALID DATA
        X_valid = train1.loc[valid_idx,LGB_Features2]
        y_valid = train1.loc[valid_idx,'Class']

        # TRAIN MODEL
        clf = LGBMClassifier(n_estimators=800,
                             max_depth=6,
                             learning_rate=0.174, 
                             metric='cross_entropy_lambda',
                             objective=None,
                             boosting_type='gbdt',
                             num_leaves=31,
                             reg_alpha=0.0,
                             reg_lambda=0.0000091,
                             early_stopping_round=50,
                             feature_fraction=0.2,
                             class_weight='balanced')
        clf.fit(X_train[LGB_Features2], y_train,eval_set=[(X_train[LGB_Features2],y_train),(X_valid[LGB_Features2], y_valid)], verbose=100)
        ensemble[valid_idx] += clf.predict_proba(X_valid)[:,1] / BAGS
        models_final[bag].append(clf)
    
     

    
        
    print()

In [ ]:
def balance_logloss_2(y_true, y_pred):
    
    y_pred = np.stack([1-y_pred,y_pred]).T
    y_pred = np.clip(y_pred, 1e-15, 1-1e-15)
    y_pred / np.sum(y_pred, axis=1)[:, None]
    nc = np.bincount(y_true)
    
    logloss = (-1/nc[0]*(np.sum(np.where(y_true==0,1,0) * np.log(y_pred[:,0]))) - 1/nc[1]*(np.sum(np.where(y_true!=0,1,0) * np.log(y_pred[:,1])))) / 2
    
    return logloss

m = balance_logloss_2( train.Class.values, oof )
print('CV Score =',m)

m2 = balance_logloss_2( train.Class.values, oof2 )
print('CV Score =',m2)

#m3 = balance_logloss_2( train.Class.values, oof3 )
#print('CV Score =',m3)

m4 = balance_logloss_2( train.Class.values, oof4 )
print('CV Score =',m4)

m5 = balance_logloss_2( train.Class.values, oof5 )
print('CV Score =',m5)
m6 = balance_logloss_2( train.Class.values, ensemble )
print('Ensemble CV Score =',m6)

In [ ]:
#CV Score = 0.14972909410238902 for LGBM

#CV Score = 0.20616804369189054 for XGB

#CV Score = 0.24514105898024902 for TabPFN

#CV Score = 0.1841793696386792 for LR

In [ ]:
#df_oof = pd.DataFrame({'oof': oof, 'oof2': oof2, 'oof3': oof3, 'oof4': oof4, 'oof5': oof5})
df_oof = pd.DataFrame({'oof': oof, 'oof2': oof2, 'oof4': oof4, 'ensemble':ensemble})

correlation_matrix = df_oof.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
plt.hist(oof,bins=100)
plt.title('Histogram of OOF',size=20)
plt.show()

plt.hist(oof2,bins=100)
plt.title('Histogram of OOF',size=20)
plt.show()

#plt.hist(oof3,bins=100)
#plt.title('Histogram of OOF',size=20)
#plt.show()

plt.hist(oof4,bins=100)
plt.title('Histogram of OOF',size=20)
plt.show()

plt.hist(ensemble,bins=100)
plt.title('Histogram of OOF',size=20)
plt.show()

In [ ]:
"""
preds = np.zeros(len(test))
preds2 = np.zeros(len(test))
preds3 = np.zeros(len(test))
preds4 = np.zeros(len(test))


for bag in range(BAGS):
    for clf in models[bag]:
            X_test = test[FEATURES]
            preds += clf.predict_proba(X_test)[:,1] / FOLDS / BAGS
            
for bag in range(BAGS):
    for clf2 in models2[bag]:
            X_test = test[FEATURES]
            preds2 += clf2.predict_proba(X_test)[:,1] / FOLDS / BAGS
            
imputer = SimpleImputer(strategy='median')
X_test_imputed = imputer.fit_transform(X_test)
scaler = StandardScaler()
X_test_scaled = scaler.transform(X_test_imputed)
            
for bag in range(BAGS):
    for clf3 in models3[bag]:
            X_test = test[FEATURES]
            preds3 += clf3.predict_proba(X_test_scaled)[:,1] / FOLDS / BAGS
            
for bag in range(BAGS):
    for clf4 in models4[bag]:
            X_test = test[FEATURES]
            preds4 += clf4.predict_proba(X_test_scaled)[:,1] / FOLDS / BAGS
"""

In [ ]:
"""
sub = test[['Id']].copy()
sub['Class_0'] = 1-preds
sub['Class_1'] = preds
sub.to_csv('submission.csv',index=False)
sub.head()
"""

In [ ]:
df_plot = train[['Id']].copy()
df_plot['class_1'] = oof
df_plot['Class'] = train[['Class']].copy()
display(df_plot)

df_plot2 = train[['Id']].copy()
df_plot2['class_1'] = oof2
df_plot2['Class'] = train[['Class']].copy()
display(df_plot2)

#df_plot3 = train[['Id']].copy()
#df_plot3['class_1'] = oof3
#df_plot3['Class'] = train[['Class']].copy()
#display(df_plot3)

df_plot4 = train[['Id']].copy()
df_plot4['class_1'] = oof4
df_plot4['Class'] = train[['Class']].copy()
display(df_plot4)

df_plot5 = train[['Id']].copy()
df_plot5['class_1'] = ensemble
df_plot5['Class'] = train[['Class']].copy()
display(df_plot5)

In [ ]:
x2 = df_plot['class_1']
y2 = df_plot.index
colors = df_plot['Class']
plt.figure(figsize=(12, 4))
blue_points = plt.scatter(x2[colors == 0], y2[colors == 0], c='blue', cmap='bwr', s=20, alpha=0.5, zorder=1)
red_points = plt.scatter(x2[colors == 1], y2[colors == 1], c='red', cmap='bwr', s=20, alpha=0.5, zorder=2)
blue_patch = mpatches.Patch(color='blue', label='Blue')
red_patch = mpatches.Patch(color='red', label='Red')
plt.legend(handles=[blue_patch, red_patch], loc='center left', bbox_to_anchor=(0, 1.1))
plt.xlabel('class_1')
plt.ylabel('Index')
plt.title('Scatter Plot: class_1 vs Index')
plt.tight_layout()
plt.show()

x2 = df_plot2['class_1']
y2 = df_plot2.index
colors = df_plot2['Class']
plt.figure(figsize=(12, 4))
blue_points = plt.scatter(x2[colors == 0], y2[colors == 0], c='blue', cmap='bwr', s=20, alpha=0.5, zorder=1)
red_points = plt.scatter(x2[colors == 1], y2[colors == 1], c='red', cmap='bwr', s=20, alpha=0.5, zorder=2)
blue_patch = mpatches.Patch(color='blue', label='Blue')
red_patch = mpatches.Patch(color='red', label='Red')
plt.legend(handles=[blue_patch, red_patch], loc='center left', bbox_to_anchor=(0, 1.1))
plt.xlabel('class_1')
plt.ylabel('Index')
plt.title('Scatter Plot: class_1 vs Index')
plt.tight_layout()
plt.show()

#x2 = df_plot3['class_1']
#y2 = df_plot3.index
#colors = df_plot3['Class']
#plt.figure(figsize=(12, 4))
#blue_points = plt.scatter(x2[colors == 0], y2[colors == 0], c='blue', cmap='bwr', s=20, alpha=0.5, zorder=1)
#red_points = plt.scatter(x2[colors == 1], y2[colors == 1], c='red', cmap='bwr', s=20, alpha=0.5, zorder=2)
#blue_patch = mpatches.Patch(color='blue', label='Blue')
#red_patch = mpatches.Patch(color='red', label='Red')
#plt.legend(handles=[blue_patch, red_patch], loc='center left', bbox_to_anchor=(0, 1.1))
#plt.xlabel('class_1')
#plt.ylabel('Index')
#plt.title('Scatter Plot: class_1 vs Index')
#plt.tight_layout()
#plt.show()

x2 = df_plot4['class_1']
y2 = df_plot4.index
colors = df_plot4['Class']
plt.figure(figsize=(12, 4))
blue_points = plt.scatter(x2[colors == 0], y2[colors == 0], c='blue', cmap='bwr', s=20, alpha=0.5, zorder=1)
red_points = plt.scatter(x2[colors == 1], y2[colors == 1], c='red', cmap='bwr', s=20, alpha=0.5, zorder=2)
blue_patch = mpatches.Patch(color='blue', label='Blue')
red_patch = mpatches.Patch(color='red', label='Red')
plt.legend(handles=[blue_patch, red_patch], loc='center left', bbox_to_anchor=(0, 1.1))
plt.xlabel('class_1')
plt.ylabel('Index')
plt.title('Scatter Plot: class_1 vs Index')
plt.tight_layout()
plt.show


x2 = df_plot5['class_1']
y2 = df_plot5.index
colors = df_plot5['Class']
plt.figure(figsize=(12, 4))
blue_points = plt.scatter(x2[colors == 0], y2[colors == 0], c='blue', cmap='bwr', s=20, alpha=0.5, zorder=1)
red_points = plt.scatter(x2[colors == 1], y2[colors == 1], c='red', cmap='bwr', s=20, alpha=0.5, zorder=2)
blue_patch = mpatches.Patch(color='blue', label='Blue')
red_patch = mpatches.Patch(color='red', label='Red')
plt.legend(handles=[blue_patch, red_patch], loc='center left', bbox_to_anchor=(0, 1.1))
plt.xlabel('class_1')
plt.ylabel('Index')
plt.title('Scatter Plot: class_1 vs Index')
plt.tight_layout()
plt.show()

In [ ]:
predicted_values = df_plot['class_1'].values
actual_values = df_plot['Class'].values
actual_values.sort()
predicted_values.sort()

predicted_values2 = df_plot2['class_1'].values
predicted_values2.sort()

#predicted_values3 = df_plot3['class_1'].values
#predicted_values3.sort()

predicted_values4 = df_plot4['class_1'].values
predicted_values4.sort()

predicted_values5 = df_plot5['class_1'].values
predicted_values5.sort()

n = len(predicted_values)
cdf = np.arange(1, n + 1) / n

sns.set(style="whitegrid")
plt.plot(cdf, predicted_values, label='LGBM')
plt.plot(cdf, predicted_values2, label='XGB')
#plt.plot(cdf, predicted_values3, label='TabPFN')
plt.plot(cdf, predicted_values4, label='LR')
plt.plot(cdf, predicted_values5, label='Ensemble')
plt.plot(cdf, actual_values, label='Actual')
plt.xlabel("Predicted Values")
plt.ylabel("CDF")
plt.title("Cumulative Distribution Function for Predictions")
plt.legend()
plt.show()

## Inference

In [ ]:
(models5[0][10]).n_features_in_

In [ ]:
lgb_preds = np.zeros(len(test))
xgb_preds = np.zeros(len(test))
svc_preds = np.zeros(len(test))
lr_preds = np.zeros(len(test))
final_preds = np.zeros(len(test))

for bag in range(BAGS):
    for clf in models[bag]:
            lgb_preds += (clf.predict_proba(test1[LGB_Features])[:,1] / FOLDS / BAGS)
for bag in range(BAGS):   
    for clf in models2[bag]:
            X_test = test1[XGB_Features].copy()
            xgb_preds += (clf.predict_proba(X_test)[:,1] / FOLDS / BAGS)

for bag in range(BAGS):   
    for clf in models4[bag]:
            X_test = df_teste2[names].copy()
            lr_preds += (clf.predict_proba(X_test)[:,1] / FOLDS / BAGS)


for bag in range(BAGS):   
    for clf in models5[bag]:
            X_test = test1[SVC_Features].copy()
            imputer = SimpleImputer(strategy='median')
            X_test_imputed = imputer.fit_transform(X_test)
            if X_test_imputed.shape[-1]!=47:
                X_test_imputed = X_test.fillna(0)
            scaler = StandardScaler()
            X_test = scaler.fit_transform(X_test_imputed)
            svc_preds += (clf.predict_proba(X_test)[:,1] / FOLDS / BAGS)


In [ ]:
#df_teste2['lgb'] = lgb_preds
#df_teste2['xgb'] = xgb_preds
#df_teste2['lr'] = lr_preds
#df_teste2['svc'] = svc_preds

In [ ]:
test1['lgb'] = lgb_preds
test1['xgb'] = xgb_preds
test1['lr'] = lr_preds
test1['svc'] = svc_preds

In [ ]:
preds = np.zeros(len(test))

for bag in range(BAGS):
    for clf in models_final[bag]:
            preds += (clf.predict_proba(test1[LGB_Features2])[:,1] / FOLDS / BAGS)

In [ ]:
sub = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')[['Id']].copy()
sub['Class_0'] = 1-preds
sub['Class_1'] = preds
sub.to_csv('submission.csv',index=False)
sub.head()